### Librerías

In [1]:
import pandas as pd
import numpy as np
from statistics import mode

### Generador de datos.

In [2]:
def generar_datos_transaccionales(n_clientes=100_000, n_transacciones=30, probabilidad_anomalia=0.1, semilla=0, fecha_inicial="2026-06-01"):

    # Generador aleatorio.
    rng = np.random.default_rng(semilla)

    # Cambio del formato a fecha.
    fecha_inicial = pd.Timestamp(fecha_inicial)

    # Listado de establecimientos y sus probabilidades.
    establecimientos = ["Supermercado", "Restaurante", "Gasolinera", "Farmacia", "Ropa", "Tecnología", "Entretenimiento", "Transporte", "Otros"]
    factores_monto = {"Supermercado": 1.30, "Restaurante": 0.80, "Gasolinera": 1.10, "Farmacia": 0.90, "Ropa": 1.40, "Tecnología": 2.50, "Entretenimiento": 0.70, "Transporte": 0.50, "Otros": 1.00}

    # Medias de establecimientos.
    medias = np.array([0.90, 0.65, 0.45, 0.30, 0.20, 0.12, 0.04, 0.015, 0.005])

    # Se almacenan las filas del DataFrame.
    clientes = []

    for cliente in range(n_clientes):

        # Se elige los estadísticos de los montos del cliente.
        media_monto_cliente = rng.uniform(50, 1000)
        desviacion_monto_cliente = rng.uniform(20, min(media_monto_cliente/1.5, 500))
        minimo_transferencia = rng.uniform(0.5, 15)
        monto_tipico_cliente = max(minimo_transferencia, rng.normal(loc=media_monto_cliente,scale=desviacion_monto_cliente))

        # Fecha inicial de las transacciones del cliente.
        desplazamiento_inicial = rng.integers(0, 40)
        fecha_actual = fecha_inicial + pd.Timedelta(days=int(desplazamiento_inicial))

        # Se establecen las probabilidades históricas de visitas de establecimientos de cada cliente.
        posiciones_establecimientos = rng.permutation(len(establecimientos))
        pesos_establecimientos = np.zeros(len(establecimientos))
        for posicion, mu in zip(posiciones_establecimientos, medias):
            pesos_establecimientos[posicion] = max(0, rng.normal(loc=mu, scale=0.1 * mu))

        probabilidades_establecimientos = pesos_establecimientos / pesos_establecimientos.sum()

        # Listas donde se almacenarán los datos.
        fechas_hora, montos, tipos_establecimiento = [], [], []

        # Se generan datos con n cantidad de transacciones por cliente.
        for _ in range(n_transacciones):

            # Se establece fecha-hora de la transacción.
            minutos_hasta_siguiente = rng.integers(1, 2_881)
            fecha_actual += pd.Timedelta(minutes=int(minutos_hasta_siguiente))

            # Se establecen los establecimientos en los que se realizó la transacción.
            establecimiento = rng.choice(establecimientos, p=probabilidades_establecimientos)

            # Se establecen los montos de la transacción.
            variacion = rng.lognormal(mean=0, sigma=0.45)
            monto = round(monto_tipico_cliente * factores_monto[establecimiento] * variacion, 2)

            # Se agregan a las listas.
            fechas_hora.append(fecha_actual)
            montos.append(monto)
            tipos_establecimiento.append(establecimiento)

        # Se decide si habrá una anomalía.
        es_anomalia = int(rng.random() <= probabilidad_anomalia)
        tipo_anomalia = "Normal"

        # Generación de anomalías.
        if es_anomalia == 1:
            aleatorio_tipo = rng.random()

            # Anomalía de orden no congruente.
            if aleatorio_tipo <= 0.40:
                tipo_anomalia = "orden_no_congruente"
                posicion = int(rng.integers(1, n_transacciones - 1))
                posiciones_anomalas = [posicion - 1, posicion, posicion + 1]
                umbral_monto_alto = np.quantile(montos, 0.85)
                for indice in posiciones_anomalas:
                    factor_anomalo = rng.uniform(1.6, 3.0)
                    montos[indice] = round(umbral_monto_alto * factor_anomalo, 2)

            # Anomalía de tiempos de transacción cortos.
            elif aleatorio_tipo <= 0.75:
                tipo_anomalia = "transaccion_corta"
                posicion = int(rng.integers(1, n_transacciones - 1))
                fecha_central = fechas_hora[posicion]
                limite_antes = min(15, int((fecha_central - fechas_hora[posicion - 1]) / pd.Timedelta(minutes=1)))
                limite_despues = min(15, int((fechas_hora[posicion + 1] - fecha_central) / pd.Timedelta(minutes=1)))
                minutos_antes = int(rng.integers(1, limite_antes + 1))
                minutos_despues = int(rng.integers(1, limite_despues + 1))
                fechas_hora[posicion - 1] = fecha_central - pd.Timedelta(minutes=minutos_antes)
                fechas_hora[posicion + 1] = fecha_central + pd.Timedelta(minutes=minutos_despues)

            # Anomalía de establecimientos extraños del cliente.
            else:
                tipo_anomalia = "establecimiento_raro"
                indices_menores = np.argsort(probabilidades_establecimientos)[:3]
                cantidad_posiciones = rng.integers(3, 6)
                indices_elegidos = rng.choice(indices_menores, size=cantidad_posiciones, replace=True)
                anomalias = [establecimientos[indice] for indice in indices_elegidos]
                posiciones = rng.choice(len(tipos_establecimiento), size=cantidad_posiciones, replace=False)

                for posicion, anomalia in zip(posiciones, anomalias):
                    tipos_establecimiento[posicion] = anomalia
                    variacion = rng.lognormal(mean=0, sigma=0.45)                    
                    montos[posicion] = round(monto_tipico_cliente * factores_monto[anomalia] * variacion, 2)

        clientes.append({"cliente_id": cliente + 1, "fechas_hora": fechas_hora, "montos": montos, "tipos_establecimiento": tipos_establecimiento, "historico_establecimiento": probabilidades_establecimientos, "es_anomalia": es_anomalia, "tipo_anomalia": tipo_anomalia})
    return pd.DataFrame(clientes)

### Explicación del generador de datos.  
El conjunto genera por cada fila un cliente. Para cada cliente se generan 3 listas de longitud 30, las cuales tienen la fecha y hora, el monto y el tipo de establecimiento donde se realizó la transacción. A parte, también se genera una lista de longitud 9, la cual contiene la proporción histórica de los tipos de negocio donde mayor cantidad de transacciones ha ejecutado.  

- La fecha y hora nos dará información del tiempo entre transacciones, ya que cuando existen fraudes, suelen buscar gastar la mayor cantidad de dinero en el menor tiempo posible. Las anomalías generadas son en base a una posición central y sus dos posiciones vecinas, las cuales pueden tener distancia de entre 1 y 15 minutos.  
- Los montos son relevantes para el análisis, ya que saber el cambio entre cantidades transaccionadas puede ser relevante para identificar fraudes. Cada cliente tiene una media y desviación estándar elegida aleatoriamente, los negocios también tienen un factor que define cuánto más se suele gastar por el tipo de negocio. Las anomalías consisten en obtener el percentil 85 y se multiplica con un factor aleatorio uniforme entre 1.2 y 2, esto en una posición elegida al azar y sus dos posiciones vecinas.  
- El tipo de establecimiento nos puede indicar un posible fraude en caso de que presenciar establecimientos poco comunes para el cliente. Para crear estas anomalías se escoge entre 3 y 5 posiciones aleatoriamente y de los 3 tipos de negocios con menor proporción, se extrae una muestra de 3 a 5 negocios con reemplazo y se reemplaza en la lista creada en las posiciones elegidas aleatoriamente.

### Se generan los datos.

In [3]:
df = generar_datos_transaccionales()
df.head(2)

,cliente_id,fechas_hora,montos,tipos_establecimiento,historico_establecimiento,es_anomalia,tipo_anomalia
0,1,"[2026-06-08 00:16:00, 2026-06-08 06:14:00, 202...","[705.29, 628.12, 272.24, 382.26, 1109.36, 706....","[Farmacia, Farmacia, Transporte, Farmacia, Sup...","[0.16643057183060853, 0.017594963536259, 0.268...",0,Normal
1,2,"[2026-06-14 07:11:00, 2026-06-15 07:48:00, 202...","[266.04, 518.87, 266.71, 193.6, 264.71, 433.54...","[Otros, Otros, Transporte, Transporte, Otros, ...","[0.001894439046728334, 0.01438288653067728, 0....",0,Normal


In [4]:
df.tipo_anomalia.value_counts()

tipo_anomalia
Normal                  89936
orden_no_congruente      3994
transaccion_corta        3611
establecimiento_raro     2459
Name: count, dtype: int64

### Variables agregadas al conjunto de datos.

In [5]:
def variables_agregadas(datos):

    # Listas que almacenan los datos.
    promedios = []
    desviaciones = []
    maximos = []
    minimos = []
    total = []
    horas = []
    negocios = []
    modas = []

    # Estadísticos de montos.
    for x in datos['montos']:
        promedios.append(np.mean(x))
        desviaciones.append(np.std(x))
        maximos.append(np.max(x))
        minimos.append(np.min(x))
        total.append(np.sum(x))

    # Monto por hora en las últimas 5 transacciones.
    for y in datos['fechas_hora']:
        horas.append((y[-1]-y[0])/ pd.Timedelta(hours=1))

    # Moda y cantidad de establecimientos.
    for w in datos['tipos_establecimiento']:
        modas.append(mode(w))
        negocios.append(len(set(w)))

    # Se agregan las variables al DataFrame.
    datos['promedio'] = promedios
    datos['desviacion'] = desviaciones
    datos['maximo'] = maximos
    datos['minimo'] = minimos
    datos['monto_hora'] = [total[z]/horas[z] for z in range(len(horas))]
    datos['moda_establecimiento'] = modas
    datos['cantidad_establecimiento'] = negocios

    return datos

### Explicación de variables agregadas.
Se agregan estadísticos de los montos, debido a que contienen cierta información histórica condensada.  
Se agrega el monto por hora, ya que puede permitir observar anomalías de transacciones cortas.  
Se agrega la moda de los establecimientos, para saber que establecimiento visita más.  
Se agrega la cantidad de tipos de establecimientos visitados, ya que puede dar información relevante en las anomalías por establecimientos poco comunes de visitar para el ciente.

### Se agregan las variables.

In [6]:
df = variables_agregadas(df)
df.head(2)

,cliente_id,fechas_hora,montos,tipos_establecimiento,historico_establecimiento,es_anomalia,tipo_anomalia,promedio,desviacion,maximo,minimo,monto_hora,moda_establecimiento,cantidad_establecimiento
0,1,"[2026-06-08 00:16:00, 2026-06-08 06:14:00, 202...","[705.29, 628.12, 272.24, 382.26, 1109.36, 706....","[Farmacia, Farmacia, Transporte, Farmacia, Sup...","[0.16643057183060853, 0.017594963536259, 0.268...",0,Normal,879.374333,602.582962,2380.04,240.25,34.208766,Farmacia,7
1,2,"[2026-06-14 07:11:00, 2026-06-15 07:48:00, 202...","[266.04, 518.87, 266.71, 193.6, 264.71, 433.54...","[Otros, Otros, Transporte, Transporte, Otros, ...","[0.001894439046728334, 0.01438288653067728, 0....",0,Normal,346.078000,165.763599,762.78,63.60,16.192467,Otros,5


### Volviendo las listas strings, para asgurar el correcto almacenamiento.

In [7]:
df['fechas_hora'] = [[str(y) for y in x] for x in df['fechas_hora']]
df['montos'] = [[str(y) for y in x] for x in df['montos']]
df['tipos_establecimiento'] = [[str(y) for y in x] for x in df['tipos_establecimiento']]
df['historico_establecimiento'] = [[str(y) for y in x] for x in df['historico_establecimiento']]

In [8]:
df.head(2)

,cliente_id,fechas_hora,montos,tipos_establecimiento,historico_establecimiento,es_anomalia,tipo_anomalia,promedio,desviacion,maximo,minimo,monto_hora,moda_establecimiento,cantidad_establecimiento
0,1,"[2026-06-08 00:16:00, 2026-06-08 06:14:00, 202...","[705.29, 628.12, 272.24, 382.26, 1109.36, 706....","[Farmacia, Farmacia, Transporte, Farmacia, Sup...","[0.16643057183060853, 0.017594963536259, 0.268...",0,Normal,879.374333,602.582962,2380.04,240.25,34.208766,Farmacia,7
1,2,"[2026-06-14 07:11:00, 2026-06-15 07:48:00, 202...","[266.04, 518.87, 266.71, 193.6, 264.71, 433.54...","[Otros, Otros, Transporte, Transporte, Otros, ...","[0.001894439046728334, 0.01438288653067728, 0....",0,Normal,346.078000,165.763599,762.78,63.60,16.192467,Otros,5


### Se genera el csv para almacenar los datos.

In [9]:
df.to_csv('datos_sinteticos.csv', index=False, sep=';')